# Konszenzus vs. információtartalom\n
\n
Projektváz motívumkeresési összehasonlító kísérlethez.

## Cél\n
Két fitneszfüggvény (konszenzus vs. információtartalom) azonos keretben történő összevetése.

## Paraméterek (kitöltendő)\n
- szekvenciák száma: 15\n
- motívumhossz: 8\n
- futások száma változatonként: >= 20\n
- azonos kiértékelési költségvetés

## Pszeudokód helye a projektben
A megadott pszeudokód a genetikus algoritmus fő ciklusát írja le (populáció inicializálás, elitizmus, szelekció, keresztezés, mutáció, újrakiértékelés, legjobb egyed visszaadása).

> P ← μ véletlen egyed; költségek kiértékelése
ISMÉTELD G-szer
    új ← a legjobb ELITE egyed MÁSOLATA
    AMÍG |új| < μ
        a, b ← versengő szelekció (k)
        c ← keresztezés(a, b) pc eséllyel, KÜLÖNBEN a másolata
        c ← mutáció(c)
        új ← új ∪ {c}
    P ← új; kiértékelés
VISSZAAD a legjobb

Ez a notebookban a GA váz részhez tartozik, konkrétan a versengő szelekció (tournament), keresztezés (crossover), mutáció (mutate) és a run ciklus megértéséhez szükséges. A projektben ez biztosítja az azonos optimalizációs keretet, amelyben csak a fitneszfüggvény változik a korrekt összehasonlításhoz.

In [ ]:
# TODO: Importok és globális beállítások

# ==== P-H09 · Konszenzus vs. információtartalom ====
# Colab-csonk – a TODO részeket neked kell kidolgozni. Az ellenőrző cellát ne töröld!
import math, random, time
import numpy as np
import matplotlib.pyplot as plt
SEED = 42
EXPECTED = """Az információtartalom-alapú fitnesz legalább olyan gyakran találja meg a motívumot, mint a konszenzus-pontszám; A/T-gazdag háttérnél jobb."""

In [ ]:
# TODO: Szintetikus adatgenerálás (15 szekvencia, beültetett 8-mer)

# ---- Feladat: Motívumkeresés (15 szekvencia, beültetett 8-mer) ----
KIND = "int"
_r = random.Random(11)
MOTIF, T, LEN, K = "TATAAGGC", 15, 60, 8
SEQS = []
for _ in range(T):
    s = [_r.choice("ACGT") for _ in range(LEN)]
    m = [c if _r.random() > 0.1 else _r.choice("ACGT") for c in MOTIF]
    p = _r.randrange(LEN - K + 1)
    s[p:p + K] = m
    SEQS.append("".join(s))
N, VALUES = T, LEN - K + 1       # gén = kezdőpozíció szekvenciánként
def kmers(pos):
    return [s[p:p + K] for s, p in zip(SEQS, pos)]
def consensus(pos):
    return "".join(max("ACGT", key=col.count) for col in zip(*kmers(pos)))
def cost(pos):
    return -sum(max(col.count(c) for c in "ACGT") for col in zip(*kmers(pos)))
def random_solution(rng):
    return [rng.randrange(VALUES) for _ in range(N)]
OPTIMUM = None                    # elvárt: consensus(best) ≈ MOTIF

In [ ]:
# TODO: Két fitneszfüggvényes változat futtatása azonos vázzal

# ---- GA váz (minimalizálás) ----
POP, GENS, PC, ELITE, TOUR_K = 60, 300, 0.9, 1, 3
PM = 1.0 / N

def tournament(pop, costs, rng, k=TOUR_K):
    # TODO: k véletlen index, a legkisebb költségű egyed MÁSOLATA
    raise NotImplementedError

def crossover(a, b, rng):
    if KIND == "perm":
        # TODO: OX vagy PMX – az utód érvényes permutáció legyen!
        raise NotImplementedError
    # TODO: bin/int: egypontos vagy uniform; real: köztes rekombináció
    raise NotImplementedError

def mutate(x, rng):
    x = x[:]
    if KIND == "bin":
        pass   # TODO: bitflip PM eséllyel
    elif KIND == "int":
        pass   # TODO: génenként PM eséllyel új érték a range(VALUES)-ból
    elif KIND == "real":
        pass   # TODO: Gauss-mutáció + vágás [LO[i], HI[i]]
    else:
        pass   # TODO: inverzió (TSP) vagy csere/beszúrás (ütemezés)
    return x

def run(seed):
    rng = random.Random(seed)
    pop = [random_solution(rng) for _ in range(POP)]
    costs = [cost(x) for x in pop]
    hist = []
    for g in range(GENS):
        order = sorted(range(POP), key=lambda i: costs[i])
        new = [pop[i][:] for i in order[:ELITE]]
        while len(new) < POP:
            a, b = tournament(pop, costs, rng), tournament(pop, costs, rng)
            c = crossover(a, b, rng) if rng.random() < PC else a[:]
            new.append(mutate(c, rng))
        pop = new
        costs = [cost(x) for x in pop]
        hist.append(min(costs))
    best = pop[min(range(POP), key=lambda i: costs[i])]
    return best, hist

In [ ]:
# TODO: >=20 futás/variáns, azonos seed-lista, azonos költségvetés

# ---- Kísérlet: független futások ----
RUNS = 10                         # Haladó szinttől: 20, Nehéz/Expert: 30
finals, curves = [], []
for s in range(RUNS):
    best, hist = run(SEED + s)
    finals.append(min(hist)); curves.append(hist)
print("medián:", np.median(finals), "| legjobb:", min(finals), "| legrosszabb:", max(finals))
for h in curves:
    plt.plot(h, alpha=0.4)
plt.xlabel("iteráció / generáció"); plt.ylabel("legjobb költség"); plt.title("Tanulási görbék"); plt.show()

In [ ]:
# TODO: Statisztika (medián, IQR) és boxplot készítése

# ---- Elvárt eredmény ellenőrzése (a jelentésbe is!) ----
print("ELVÁRT:", EXPECTED)
# TODO: táblázat: elvárt | kapott | teljesül-e (igen/nem) | magyarázat

## Értelmezés (5-8 mondat)\n
- TODO: Eredmények rövid szöveges értelmezése